# OpenRouter model screening before Tinker training

One secret-carrying answer per APPS problem/model, at most **100 shared problems**.
Compare functional, exact-message, and joint pass@1. Inference uses OpenRouter, with a Codex Luna baseline below;
later training would be on Tinker. This notebook does not train models.

**Preparation is free of inference calls.** It saves all requests and estimates cost.
The later execution cell asks yes/no before spending on OpenRouter and Modal.
See [README.md](README.md) for model availability, limitations, and artifact schemas.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic_yaml import parse_yaml_raw_as

# Locate the repository when Jupyter starts in this notebook's subdirectory.
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / "ciphers/variable_naming_in_python_v2").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from ciphers.variable_naming_in_python_v2.data.apps import REPO_ROOT
from ciphers.variable_naming_in_python_v2.data.codex_apps import SecretTask
from ciphers.variable_naming_in_python_v2.decoder import CipherConfig
from ciphers.variable_naming_in_python_v2.tinker.codex_evaluate import prepare_codex_comparison, run_codex_comparison
from ciphers.variable_naming_in_python_v2.tinker.openrouter_evaluate import run_prepared, summarize
from ciphers.variable_naming_in_python_v2.tinker.openrouter_prepare import RunConfig, artifact_path, prepare_run

if not all(os.environ.get(key) for key in ("OPENROUTER_API_KEY", "STEGO_ARTIFACTS_DIR")):
    env_path = REPO_ROOT / ".env"
    if not env_path.is_file():
        raise FileNotFoundError("Set the environment variables or provide the repository-root .env")
    load_dotenv(env_path, override=False)
for key in ("OPENROUTER_API_KEY", "STEGO_ARTIFACTS_DIR"):
    if not os.environ.get(key):
        raise RuntimeError(f"Missing required environment variable: {key}")

In [ ]:
cipher_path = REPO_ROOT / "ciphers/variable_naming_in_python_v2/tinker/official_cipher.yaml"
secret = SecretTask(
    cipher=parse_yaml_raw_as(CipherConfig, cipher_path.read_text()),
    message_bits="101",
)
config = RunConfig(secret=secret, num_problems=100, max_tokens=16384, estimated_output_tokens=4096, timeout_s=900)
config

## Prepare and review

The next cell downloads the filtered APPS dataset and public model prices, writes all
requests, then costs the saved prompts. It sends **no inference requests**.
To review an existing run, replace the prepare call with `Path("variable_naming_v2/tinker/<run-id>")`.

The estimate assumes 4,096 output tokens per answer; the limit scenario assumes
16,384, including reasoning where the provider includes it in the limit.
Neither is a hard spending ceiling. Modal grading and account fees are excluded.
The shared request deadline is 900 seconds to accommodate slower reasoning answers.

**To resume:** set `run_dir = Path("variable_naming_v2/tinker/<existing-run-id>")`
in the next cell instead of calling `prepare_run(config)`. After a kernel restart,
run the import/setup cells first. Use this same path in the provider section below.

In [ ]:
run_dir = prepare_run(config)
print("Run directory:", artifact_path(run_dir))
estimate = json.loads((artifact_path(run_dir) / "estimate.json").read_text())
price_rows = []
for row in estimate["models"]:
    pricing = row["pricing"]
    price_rows.append({
        "model": row["model"], "available": row["available"], "requests": row["requests"],
        "input_characters": row["input_characters"], "estimated_input_tokens": row["input_tokens"],
        "input_USD_per_M": pricing["prompt"] * 1e6 if pricing else None,
        "output_USD_per_M": pricing["completion"] * 1e6 if pricing else None,
        "estimated_USD": row["estimated_usd"], "limit_scenario_USD": row["limit_scenario_usd"],
    })
display(pd.DataFrame(price_rows))
print(f"Estimated inference cost: ${estimate['estimated_usd']:.2f}")
print(f"Output-limit scenario: ${estimate['limit_scenario_usd']:.2f}")
print("Inspect requests.jsonl, grading_cases.json, and config.json in the run directory.")

## Preview saved prompts

Randomly sample three request rows from the saved `requests.jsonl` file and display
their actual prompts, with model and problem IDs. The seed makes the sample repeatable;
different model rows can contain the same problem. This cell makes no API calls.

In [ ]:
request_rows = [
    json.loads(line)
    for line in (artifact_path(run_dir) / "requests.jsonl").read_text().splitlines()
]
sampled_requests = random.Random(config.seed).sample(request_rows, k=min(3, len(request_rows)))
for request_row in sampled_requests:
    display(Markdown(f"### APPS {request_row['problem_id']} — {request_row['body']['model']}"))
    for message in request_row["body"]["messages"]:
        display(Markdown(message["content"]))

## Stopping and resuming

Answers and grades are saved incrementally. A normal notebook interrupt stops new
claims and waits for active workers to finish saving; a forced kernel restart can
lose unsaved in-flight answers but preserves flushed files.

After stopping the old invocation, reuse the same saved `run_dir` and set
`resume=True` in the OpenRouter cell or `codex_resume=True` in the Codex cell.
Completed grades are skipped, saved answers are graded, and only missing answers
are generated. Codex also recovers SDK `answer.json` checkpoints. A fully completed
run makes no remote calls. You may change worker count; preserve all task/model
inputs and grading limits. Use only one invocation per run directory.

Unsaved remote answers may require another billed call. Corrupt/inconsistent cache
files raise an error for inspection rather than being silently discarded.

## Optional paid execution

Run only after reviewing the saved requests and estimate. Answer `yes` to start;
anything else sends no inference requests. Execution uses the selected worker count, with one answer
per problem/model, and saves results incrementally. An infrastructure error stops
the run. Use `resume=True` to continue a stopped run from saved outputs.

In [ ]:
num_workers = 16  # Set to 1 for sequential execution, or 32 for more concurrency.
resume = False  # Set True to continue this same saved run after stopping it.
answer = input(f"Run {sum(row['requests'] for row in estimate['models'])} saved requests? "
               f"Estimate ${estimate['estimated_usd']:.2f}; output-limit scenario "
               f"${estimate['limit_scenario_usd']:.2f}, plus Modal. Type yes/no: ").strip().lower()
if answer == "yes":
    results = run_prepared(run_dir, approved=True, num_workers=num_workers, resume=resume)
else:
    print("No inference requests sent.")

In [ ]:
# Safe to run before inference or after an interrupted run; incomplete rates stay blank.
display(pd.DataFrame(summarize(run_dir)))

## Codex Luna on the same saved inputs

Use the prepared `run_dir` above, even if you skip OpenRouter inference. This
copies one prompt per problem into `codex-luna/`, preserving the exact user text,
cipher, message, and private grading cases. There is no resampling or prompt repair.

Luna uses the existing ChatGPT-backed Codex login with tools disabled and a fresh
thread per answer. The shared threaded runner applies the same output parser,
Modal evaluator, and decoder. Codex receives prompt-driven JSON instructions,
without API schema enforcement.

**Provider differences:** Codex has fixed no-tools base instructions, its own
reasoning/sampling defaults, and no equivalent output-token cap in this helper.
These are recorded in `comparison.json`; only the task inputs and scoring are
strictly matched. Older OpenRouter runs using an earlier cipher are not comparable.

In [ ]:
codex_run_dir = run_dir / "codex-luna"
if not artifact_path(codex_run_dir).exists():
    codex_run_dir = prepare_codex_comparison(run_dir)
codex_num_workers = 16
print("Luna inputs:", artifact_path(codex_run_dir))

### Run Luna (subscription usage and Modal grading)

Running the next cell explicitly starts the 100-problem Luna comparison. It does
not send OpenRouter requests. Responses and grading results are saved incrementally;
set `codex_resume=True` to continue a stopped run from saved outputs.

In [ ]:
codex_resume = False  # Set True to reuse saved answers and grades.
codex_run_dir = run_codex_comparison(run_dir, approved=True, num_workers=codex_num_workers, resume=codex_resume)
display(pd.DataFrame(summarize(codex_run_dir)))

In [ ]:
# Rates stay blank for models whose matching prepared run has not completed.
display(pd.DataFrame(summarize(run_dir) + summarize(codex_run_dir)))

## Recorded Luna result

Run dated 2026-09-16 UTC: **100 first completed answers**, generated
with **16 workers** for the same 100 APPS problems, **77-group official cipher**,
four length bits, and payload `101`.

| Metric | Result |
| --- | --- |
| All supplied code tests passed | **79 passed, 20 failed, 1 grading error** |
| Exact secret message recovered | **87/100 (87%)** |
| Both conditions | **69/100 (69%)** |

Functional success over all 100 is bounded by **79–80%**;
joint success is exact because the ungraded answer also fails message decoding.
The unresolved answer, APPS 4039, repeatedly kills the
Modal evaluator with exit code 137. Its code constructs lists over extremely large
ranges, suggesting memory exhaustion; the error remains an infrastructure/runner
error under the existing evaluator contract. It is not silently counted as a
failed unit test. `summarize()` leaves rates blank because only 99 grades completed.

The first run stopped with a 300-second SDK timeout after saving 84 grades and 85
answers. Recovery retained all completed answers, retried grading the saved answer,
started 14 previously unclaimed problems, and retried the one SDK call without a
final answer using a 900-second deadline. **101 SDK generation attempts yielded
100 final answers**; no completed answer was regenerated or repaired. There were
**0** malformed/empty/truncated final answers. The notebook
now uses a shared 900-second deadline for future provider comparisons.

Artifacts relative to `STEGO_ARTIFACTS_DIR`:
`variable_naming_v2/tinker/20260915T235628Z-b76630f2/codex-luna-comparison`.
`summary.json` contains counts and bounds; `recovery.json` links the preserved
original and recovery runs; `ungraded_answers.json` records the unresolved grade
and its independently decoded message. The parent directory contains matching
OpenRouter requests; no OpenRouter inference was run for this cipher. Older
OpenRouter results use different cipher settings and are excluded.

Task inputs and scoring are shared. Codex's fixed base instructions, native
sampling/reasoning defaults, and lack of the OpenRouter output-token cap remain
provider differences, so this is a practical screening baseline.


In [ ]:
recorded_codex_run = Path("variable_naming_v2/tinker/20260915T235628Z-b76630f2/codex-luna-comparison")
recorded_report = json.loads((artifact_path(recorded_codex_run) / "summary.json").read_text())
display(recorded_report)

## Single-request latency: OpenRouter versus Codex

Measure **10 actual saved problem prompts per provider**, with **one request in
flight at a time** and no Modal grading. Both providers get the same sampled
prompts. Their order alternates by problem to reduce ordering bias. Let other
inference runs finish first if you want an isolated measurement.

This measures **full response latency**, not network ping or time to first token.
OpenRouter includes HTTP connection setup, generation, and response download;
Codex includes the existing helper's preflight, SDK startup, generation, and SDK
artifact writes. Each Codex call uses a fresh thread, as in the main evaluator.
The two backends can use different reasoning/sampling settings, and the Codex
helper does not enforce OpenRouter's output-token limit. Output character counts
are included to help interpret differences in answer length.

The next cells only prepare inputs and retrieve public pricing. They save all
20 requests before the explicit execution cell. Edit the OpenRouter model below;
it must be present in the source run. To use an existing run after restarting the
kernel, run the import/setup cells and set `run_dir` to its saved relative path.
No 100-problem inference cell needs to be run.

In [ ]:
from datetime import datetime, timezone
from time import perf_counter
from uuid import uuid4

from pydantic import BaseModel, ConfigDict, Field

from ciphers.variable_naming_in_python_v2.data.codex_apps import CodexInferenceConfig, infer
from ciphers.variable_naming_in_python_v2.tinker.codex_evaluate import LUNA_MODEL
from ciphers.variable_naming_in_python_v2.tinker.openrouter_evaluate import ChatResponse, send_request
from ciphers.variable_naming_in_python_v2.tinker.openrouter_prepare import PreparedRequest, estimate_cost, fetch_catalog


class LatencyConfig(BaseModel):
    """Select saved input prompts; all source generation settings remain unchanged."""

    model_config = ConfigDict(extra="forbid", frozen=True)
    source_run: Path
    openrouter_model: str = "openai/gpt-oss-20b"
    num_samples: int = Field(default=10, ge=1, le=100, strict=True)
    seed: int = 42


latency_config = LatencyConfig(source_run=run_dir)
latency_config

In [ ]:
latency_source = artifact_path(latency_config.source_run)
latency_source_config = RunConfig.model_validate_json((latency_source / "config.json").read_text())
latency_candidates = [
    request for line in (latency_source / "requests.jsonl").read_text().splitlines()
    if (request := PreparedRequest.model_validate_json(line)).body.model == latency_config.openrouter_model
]
if len({request.problem_id for request in latency_candidates}) != len(latency_candidates):
    raise ValueError("Expected one saved request per problem for the selected model")
if len(latency_candidates) < latency_config.num_samples:
    raise ValueError("Choose a model with enough requests in the source requests.jsonl")
latency_sample = random.Random(latency_config.seed).sample(latency_candidates, latency_config.num_samples)
latency_jobs = []
for index, request in enumerate(latency_sample):
    providers = ("openrouter", "codex") if index % 2 == 0 else ("codex", "openrouter")
    for provider in providers:
        latency_jobs.append({"provider": provider, "request": request.model_dump(mode="json")})

latency_run_dir = latency_config.source_run / "latency" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
latency_directory = artifact_path(latency_run_dir)
latency_directory.mkdir(parents=True, exist_ok=False)
latency_codex_config = CodexInferenceConfig(
    model=LUNA_MODEL, timeout_s=latency_source_config.timeout_s,
    artifact_subdir=latency_run_dir / "codex-generation",
)
(latency_directory / "config.json").write_text(latency_config.model_dump_json(indent=2))
(latency_directory / "source_config.json").write_text(latency_source_config.model_dump_json(indent=2))
(latency_directory / "codex_config.json").write_text(latency_codex_config.model_dump_json(indent=2))
(latency_directory / "requests.jsonl").write_text("".join(json.dumps(job) + "\n" for job in latency_jobs))

latency_catalog = fetch_catalog((latency_config.openrouter_model,))
if latency_config.openrouter_model not in latency_catalog:
    raise ValueError("Selected model is no longer available on OpenRouter")
latency_cost_config = latency_source_config.model_copy(update={
    "models": (latency_config.openrouter_model,), "num_problems": latency_config.num_samples,
})
latency_estimate = estimate_cost(latency_sample, latency_cost_config, latency_catalog)[0]
(latency_directory / "estimate.json").write_text(latency_estimate.model_dump_json(indent=2))
print("Latency artifacts:", latency_directory)
print("Sampled problem IDs:", [request.problem_id for request in latency_sample])
print(f"OpenRouter estimate: ${latency_estimate.estimated_usd:.4f}; output-limit scenario: ${latency_estimate.limit_scenario_usd:.4f}")
print(f"Plus {latency_config.num_samples} Codex subscription calls; no Modal grading.")

### Run the sequential timing sample

This cell asks **yes/no** before making new calls. Timing includes the first call;
there are no discarded warmups or automatic retries. Failures are timed and saved
separately rather than counted as fast successful responses. Each completed attempt
is flushed to disk, and exclusive creation prevents accidentally rerunning the
same benchmark. Prepare a new benchmark directory to repeat it.

The saved `requests.jsonl` rows contain `provider` plus the original
`PreparedRequest` under `request`. OpenRouter sends that saved body; Codex uses
its user prompt with `LUNA_MODEL` and the saved `codex_config.json`.
`timings.jsonl` records provider/model/problem, UTC start time, elapsed seconds,
status, finish reason, output characters, nullable error, and the full response
(OpenRouter raw JSON or the Codex `InferenceResult`).

In [ ]:
async def measure_latency(job: dict) -> dict:
    """Time one saved job through the final answer, excluding grading and JSONL writes.

    job requires provider (openrouter/codex) and request (PreparedRequest JSON).
    Uses the prepared latency_source_config deadline and latency_codex_config.
    Returns provider, model, problem_id, started_at, elapsed_s, status (completed,
    unusable_response, or error), finish_reason, output_chars, error, and response.
    Errors retain their elapsed time; no retries or code-quality checks occur.
    """
    provider = job["provider"]
    request = PreparedRequest.model_validate(job["request"])
    row = {
        "provider": provider, "model": request.body.model if provider == "openrouter" else LUNA_MODEL,
        "problem_id": request.problem_id, "started_at": datetime.now(timezone.utc).isoformat(),
        "status": "error", "finish_reason": None, "output_chars": None, "error": None, "response": None,
    }
    started = perf_counter()
    try:
        if provider == "openrouter":
            raw = send_request(request, latency_source_config.timeout_s)
        elif provider == "codex":
            answer = await infer(request.body.messages[0].content, latency_codex_config, response_format="text")
            raw = answer.model_dump(mode="json")
        else:
            raise ValueError(f"Unknown provider: {provider}")
        row["elapsed_s"] = perf_counter() - started
        row["response"] = raw
        if provider == "openrouter":
            parsed = ChatResponse.model_validate(raw)
            if parsed.error is not None or len(parsed.choices) != 1:
                raise ValueError(f"Invalid API completion: {parsed.error or 'expected one choice'}")
            content = parsed.choices[0].message.content or ""
            row["finish_reason"] = parsed.choices[0].finish_reason
        else:
            content = answer.text
            row["finish_reason"] = "stop"
        row["output_chars"] = len(content)
        row["status"] = "completed" if row["finish_reason"] == "stop" and content.strip() else "unusable_response"
    except Exception as error:
        row.setdefault("elapsed_s", perf_counter() - started)
        row["error"] = f"{type(error).__name__}: {error}"
    return row


latency_approval = input(f"Run {len(latency_jobs)} NEW sequential timing requests? OpenRouter estimate "
                         f"${latency_estimate.estimated_usd:.4f}, plus Codex subscription usage. Type yes/no: ").strip().lower()
if latency_approval == "yes":
    with (latency_directory / "timings.jsonl").open("x") as latency_file:
        for line in (latency_directory / "requests.jsonl").read_text().splitlines():
            latency_row = await measure_latency(json.loads(line))
            latency_file.write(json.dumps(latency_row) + "\n")
            latency_file.flush()
            print(f"{latency_row['provider']:10} APPS {latency_row['problem_id']}: "
                  f"{latency_row['elapsed_s']:.1f}s ({latency_row['status']})")
else:
    print("No timing inference requests sent.")

### Latency statistics

Seconds below describe **completed, nonempty final responses**; code correctness
and message encoding are not graded here. Attempts and failures are shown alongside
those statistics, including for an interrupted benchmark. Mean, median, p90, minimum,
maximum, and sample standard deviation summarize the observed full-call latencies.
With only 10 samples per provider, p90 is descriptive and highly uncertain; it is
not a service-level guarantee. All timings include SDK/network/backend waiting and
cannot isolate server-side queueing from generation.

In [ ]:
latency_results_path = latency_directory / "timings.jsonl"
if not latency_results_path.exists() or not latency_results_path.read_text().strip():
    print("No saved timing samples yet.")
else:
    latency_frame = pd.DataFrame(json.loads(line) for line in latency_results_path.read_text().splitlines())
    latency_summary_rows = []
    for provider, model in (("openrouter", latency_config.openrouter_model), ("codex", LUNA_MODEL)):
        attempts = latency_frame[latency_frame["provider"] == provider]
        completed = attempts[attempts["status"] == "completed"]
        seconds = completed["elapsed_s"]
        latency_summary_rows.append({
            "provider": provider, "model": model, "planned": latency_config.num_samples,
            "attempted": len(attempts), "completed": len(completed),
            "errors": int((attempts["status"] == "error").sum()),
            "unusable_responses": int((attempts["status"] == "unusable_response").sum()),
            "mean_s": seconds.mean(), "median_s": seconds.median(), "p90_s": seconds.quantile(0.9),
            "min_s": seconds.min(), "max_s": seconds.max(), "std_s": seconds.std(),
            "mean_output_chars": completed["output_chars"].mean(),
        })
    latency_summary = pd.DataFrame(latency_summary_rows)
    display(latency_summary.round(2))
    display(latency_frame[["provider", "model", "problem_id", "elapsed_s", "status", "output_chars", "error"]])
    (latency_directory / "summary.json").write_text(latency_summary.to_json(orient="records", indent=2))